In [ ]:
import os, glob, gc, re, sys
import numpy as np
import xarray as xr
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ✅ 导入计算函数
try:
    # 优先尝试从当前目录导入
    from aostools_functions import ComputeEPfluxDiv
except ImportError:
    # 备选：从 /mnt/data 导入
    sys.path.insert(0, "/mnt/data")
    from aostools_functions import ComputeEPfluxDiv

# ============================================================
# MERRA-2 路径配置
# ============================================================
ROOT_DIR = "/mnt/soclim0/public_data/weiji/MERRA2_Processed"
U_DIR     = os.path.join(ROOT_DIR, "U")
V_DIR     = os.path.join(ROOT_DIR, "V")
T_DIR     = os.path.join(ROOT_DIR, "T")
# OMEGA 暂时不使用，保留路径定义以备未来整合
OMEGA_DIR = os.path.join(ROOT_DIR, "OMEGA") 

OUT_DIR     = os.path.join(ROOT_DIR, "EPflux_daily")
# 文件名标记为无 Omega 版本以示区别
OUT_FILE_NC = os.path.join(OUT_DIR, "MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- Settings ----------------
LAT_RANGE   = (40.0, 80.0)
DO_UBAR     = False
WAVE        = -1
MAX_WORKERS = 4

# ---------------- Helpers ----------------
def file_for_merra(var_dir, year_str, var_name):
    return os.path.join(var_dir, f"MERRA2.{var_name}.{year_str}.nc")

def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel({lat_name: slc})
    return out

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

# ---------------- Single-year worker (MERRA-2) ----------------
def process_one_year_merra(year_str):
    """
    MERRA-2 已经是 Pressure Levels。
    已调整：如果 OMEGA 不存在，则 w 传入 None。
    """
    try:
        fU = file_for_merra(U_DIR, year_str, "U")
        fV = file_for_merra(V_DIR, year_str, "V")
        fT = file_for_merra(T_DIR, year_str, "T")
        
        # 基础文件检查
        for fp in [fU, fV, fT]:
            if not os.path.exists(fp):
                return f"⚠️ Missing core file for {year_str}: {fp}"

        # 可选：检查 OMEGA 是否存在
        fW = file_for_merra(OMEGA_DIR, year_str, "OMEGA")
        has_w = os.path.exists(fW)

        with xr.open_dataset(fU) as dsU, \
             xr.open_dataset(fV) as dsV, \
             xr.open_dataset(fT) as dsT:

            if dsU.sizes.get("time", 0) < 3: return None

            u_np = dsU["U"].values
            v_np = dsV["V"].values
            t_np = dsT["T"].values
            
            # 处理 w 变量
            if has_w:
                with xr.open_dataset(fW) as dsW:
                    # MERRA-2 OMEGA (Pa/s) -> hPa/s
                    w_np = dsW["OMEGA"].values / 100.0
            else:
                # 如果没有 OMEGA 文件，显式传递 None
                w_np = None
            
            lat_np = dsU["lat"].values
            pres_hpa = dsU["lev"].values

            # 计算 EP flux (ep2 是 Fz)
            # 当 w 为 None 时，ComputeEPfluxDiv 内部仅利用 T, U, V 计算 ep2_cart
            _, ep2, _, _ = ComputeEPfluxDiv(
                lat=lat_np,
                pres=pres_hpa,
                u=u_np,
                v=v_np,
                t=t_np,
                w=w_np,
                do_ubar=DO_UBAR,
                wave=WAVE
            )

            da_ep2 = xr.DataArray(
                ep2,
                coords={"time": dsU["time"], "plev": pres_hpa, "lat": lat_np},
                dims=("time", "plev", "lat"),
                name="Fz"
            )

            # 40–80N 面积加权平均
            da_sub = sel_latband(da_ep2, LAT_RANGE[0], LAT_RANGE[1], "lat")
            out = coslat_weighted_mean(da_sub).compute()
            out.name = "Fz"

        gc.collect()
        return out

    except Exception as e:
        return f"Error in {year_str}: {str(e)}"

# ---------------- Main ----------------
def main():
    u_files = sorted(glob.glob(os.path.join(U_DIR, "MERRA2.U.*.nc")))
    years = [os.path.basename(f).split('.')[2] for f in u_files]
    
    if not years:
        print(f"❌ No files found in {U_DIR}")
        return

    print(f"🚀 MERRA2 EPflux: {len(years)} years, workers={MAX_WORKERS}")
    pool_results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        fut = {ex.submit(process_one_year_merra, y): y for y in years}
        for f in tqdm(as_completed(fut), total=len(fut), desc="MERRA2 Parallel"):
            res = f.result()
            if isinstance(res, xr.DataArray):
                pool_results.append(res)
            elif res: print(f"⚠️ {res}")

    if pool_results:
        Fz_all = xr.concat(pool_results, dim="time").sortby("time")
        ds_out = xr.Dataset({"Fz": Fz_all})
        ds_out.attrs = {
            "description": "MERRA-2 EP Flux (Fz) 40-80N mean", 
            "units": "m^2/s^2",
            "method": "Calculated without OMEGA (w=None)"
        }
        ds_out.to_netcdf(OUT_FILE_NC)
        print(f"✅ MERRA2 Done: {OUT_FILE_NC}")

if __name__ == "__main__":
    main()